In [ ]:
"""
RAG (Retrieval-Augmented Generation) Pipeline Demonstration
-----------------------------------------------------------
This script demonstrates how to build a complete RAG system from scratch.
It ingests a text file, breaks it into chunks, converts those chunks into
mathematical vectors (embeddings), stores them in a database, and then uses
a Large Language Model (Qwen) to answer questions based *only* on that data.
"""

# ==========================================
# INSTALLATION
# ==========================================
# !pip install langchain_community
# !pip install chromadb
# !pip install sentence_transformers
# !pip install transformers

# ==========================================
# IMPORTS
# ==========================================
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import pipeline

# ==========================================
# STEP 1: DATA INGESTION
# ==========================================
print("--- Step 1: Loading Document ---")
# We load a local markdown file containing our custom knowledge base (tennis facts).
loader = TextLoader("/content/sample_data/tennis_details.md")
text_doc = loader.load()
# text_doc[0].page_content contains the raw string of the entire document.


# ==========================================
# STEP 2: CHUNKING / TEXT SPLITTING
# ==========================================
print("--- Step 2: Chunking Data ---")
# LLMs have context limits, and vector searches work best on small, focused paragraphs.
# We split the long document into smaller chunks based on Markdown headers (##).
split_condition = [("##", "title")]
splitter = MarkdownHeaderTextSplitter(split_condition)
doc_splits = splitter.split_text(text_doc[0].page_content)

# Extract just the text content from the split objects into a clean list
text_chunks = [split.page_content for split in doc_splits]

print(f"Total chunks created: {len(text_chunks)}")
# EXPECTED OUTPUT: 7


# ==========================================
# STEP 3: GENERATE EMBEDDINGS (Mathematical Representation)
# ==========================================
print("--- Step 3: Generating Embeddings ---")
# We use a lightweight sentence-transformer model (all-MiniLM-L6-v2) to convert
# human-readable text into high-dimensional arrays of numbers (vectors).
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_chunk(chunk):
    """Converts a text string into a normalized mathematical vector."""
    return embedding_model.encode([chunk], normalize_embeddings=True)

# Let's test it on our second chunk of text to see what a vector looks like
sample_embedding = embed_chunk(text_chunks[1]).tolist()[0]
print(f"Embedding length (Dimensions): {len(sample_embedding)}")
# EXPECTED OUTPUT: 384 dimensions (The specific length for MiniLM-L6)


# ==========================================
# STEP 4: VECTOR DATABASE (ChromaDB)
# ==========================================
print("--- Step 4: Storing in Vector DB ---")
# We need a special database to store and quickly search through our numerical vectors.
# We pass our text chunks and the embedding model into ChromaDB.
vector_db = Chroma.from_texts(
    text_chunks,
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    persist_directory="/tmp/chroma_db"
)


# ==========================================
# STEP 5: INITIALIZE THE LLM (Generative Engine)
# ==========================================
print("--- Step 5: Loading the LLM ---")
# We load Qwen2.5-1.5B-Instruct. This model will take our retrieved data and
# format it into a natural, conversational answer.
pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")


# ==========================================
# STEP 6: CORE RAG LOGIC (Retrieve & Generate)
# ==========================================
def retrieve_and_generate(query, threshold=1):
    """
    Core RAG Function:
    1. Searches the database for text chunks similar to the user's question.
    2. Filters out bad matches using a threshold score.
    3. Injects the good match into a prompt.
    4. Asks the LLM to read the prompt and generate an answer.
    """
    # 1. Similarity Search: Find the top 1 (k=1) most relevant chunk from our database
    search_results = vector_db.similarity_search_with_score(query, k=1)

    # 2. Filter: If no results, or the distance score is too high (meaning low similarity)
    if not search_results or search_results[0][1] > threshold:
        return "I don't know the answer. There is no available context in the vector DB."

    # Extract the exact text and the similarity score
    retrieved_context = search_results[0][0].page_content
    similarity_score = search_results[0][1]

    print(f"--- RAG DIAGNOSTICS ---")
    print(f"Similarity Score: {similarity_score:.4f}")
    print(f"Retrieved Context: {retrieved_context}\n")

    # 3. Prompt Construction: Combine instructions, context, and the user's question
    prompt = f"Answer the question using the given context\nContext: {retrieved_context}\nQuestion: {query}\nAnswer: "

    # 4. Generation: Pass the massive prompt to the LLM to get a human-like answer
    response = pipe(prompt, max_new_tokens=100)

    return response[0]["generated_text"]


# ==========================================
# STEP 7: EXECUTION & TESTING
# ==========================================
print("--- Step 7: Testing the System ---")
question = "what is tennis"
response = retrieve_and_generate(question)

print("FINAL LLM OUTPUT:")
print(response)

# EXPECTED OUTPUT:
# Similarity Score: 0.2799
# Retrieved Context: Tennis is a popular sport played between two players...
#
# FINAL LLM OUTPUT:
# Answer the question using the given context
# Context: Tennis is a popular sport played between two players...
# Question: what is tennis
# Answer:  tennis is a sport in which two people play against each other, one person uses a rackets and hits the ball back and forth over a net...